In [1]:
# Discover repo root and read all CSV files from the per-series folders
from pathlib import Path
import pandas as pd
import sys

# Find repo root
repo_root = Path.cwd()
for candidate in [repo_root] + list(repo_root.parents):
    if (candidate / 'pyproject.toml').exists() or (candidate / '.git').exists():
        repo_root = candidate
        break

# Add repo root to sys.path (BEFORE the import attempt)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
    print(f'Added {repo_root} to sys.path')

# Dataset roots
wiertsema_dir = repo_root / 'output_data' / 'wiertsema'
fugro_dir = repo_root / 'output_data' / 'fugro'

print('wiertsema dataset root ->', wiertsema_dir)
print('fugro dataset root    ->', fugro_dir)

Added D:\Users\jvanruitenbeek\data_validation to sys.path
wiertsema dataset root -> D:\Users\jvanruitenbeek\data_validation\output_data\wiertsema
fugro dataset root    -> D:\Users\jvanruitenbeek\data_validation\output_data\fugro


# Loop over folder and generate report from the filtered files

In [2]:
# Loop over filtered CSV files and generate validation report

# Choose dataset: 'wiertsema' or 'fugro'
dataset_choice = 'wiertsema'

if dataset_choice.lower() == 'wiertsema':
    dataset_root = wiertsema_dir
elif dataset_choice.lower() == 'fugro':
    dataset_root = fugro_dir
else:
    raise ValueError("dataset_choice must be 'wiertsema' or 'fugro'")

# Recursive scan for validated files: <origin>/validated/*.csv
csv_files = sorted(dataset_root.glob('*/validated/*.csv'))
print(f'Found {len(csv_files)} CSV files in dataset {dataset_choice}\\n')

# Initialize list to store metrics
report_data = []

# Loop over each file
for i, csv_file in enumerate(csv_files, start=1):
    source_origin_stem = csv_file.parent.parent.name
    try:
        print(f'[{i}/{len(csv_files)}] Processing: {csv_file.name} (origin={source_origin_stem})')

        # Read the CSV
        df = pd.read_csv(
            csv_file,
            index_col=0,
            parse_dates=True,
            encoding="utf-8-sig",
            encoding_errors="replace"
        )

        # Coerce head column to numeric
        df['head'] = pd.to_numeric(df['head'], errors='coerce')
        head_series = df['head'].dropna()

        if len(head_series) == 0:
            print('  [ERR] No valid head data\\n')
            continue

        # Calculate metrics
        first_entry = head_series.index[0]
        last_entry = head_series.index[-1]
        num_entries = len(head_series)

        # Time span in days
        time_span_days = (last_entry - first_entry).days

        # Completeness: entries with values / possible entries (hourly)
        possible_entries = (time_span_days * 24) + 1
        completeness_pct = (num_entries / possible_entries * 100) if possible_entries > 0 else 0

        # Largest gap in hours
        time_diffs = head_series.index.to_series().diff().dt.total_seconds() / 3600
        largest_gap_hours = time_diffs.max() if len(time_diffs) > 1 else 0

        # Min and max values
        lowest_value = head_series.min()
        highest_value = head_series.max()

        # Largest jump
        head_diff = head_series.diff().abs()
        largest_jump = head_diff.max() if len(head_diff) > 1 else 0

        # Count filtered values for each algorithm
        filter_counts = {}
        for col in ['v1', 'v2', 'v3', 'v4']:
            if col in df.columns:
                filter_counts[col] = df[col].notna().sum()
            else:
                filter_counts[col] = 0

        report_data.append({
            'Source Origin': source_origin_stem,
            'Filename': csv_file.stem,
            'First Entry': first_entry,
            'Last Entry': last_entry,
            'Number of Entries': num_entries,
            'Completeness (%)': round(completeness_pct, 2),
            'Length (days)': time_span_days,
            'Largest Gap (hours)': round(largest_gap_hours, 2),
            'Lowest Value (m)': round(lowest_value, 4),
            'Highest Value (m)': round(highest_value, 4),
            'Largest Jump (m)': round(largest_jump, 4),
            'Filtered by v1 (count)': filter_counts['v1'],
            'Filtered by v2 (count)': filter_counts['v2'],
            'Filtered by v3 (count)': filter_counts['v3'],
            'Filtered by v4 (count)': filter_counts['v4'],
        })

        print('  [OK] Metrics calculated\\n')

    except Exception as e:
        print(f'  [ERR] Error processing {csv_file.name}: {e}\\n')

# Create DataFrame from report data
report_df = pd.DataFrame(report_data)

# Save to Excel
output_file = repo_root / 'output_data' / f'validation_report_{dataset_choice}.xlsx'
report_df.to_excel(output_file, index=False, sheet_name='Validation Report')

print(f'[OK] Report saved to: {output_file}')
print('\\nSummary Statistics:')
print(report_df.describe(include='all'))

Found 207 CSV files in dataset wiertsema\n
[1/207] Processing: 83034-1 HB001PB01 BE0049+00_BUKR_GMW_PB1_F-229.csv (origin=83034-1)
  [OK] Metrics calculated\n
[2/207] Processing: 83034-1 HB002PB01 BE0049+00_BIKR_GMW_PB1_F-227.csv (origin=83034-1)
  [OK] Metrics calculated\n
[3/207] Processing: 83034-1 HB003PB01 BE0049+00_INST_B_GMW_PB1_F-365.csv (origin=83034-1)
  [OK] Metrics calculated\n
[4/207] Processing: 83034-1 HB005PB01 BE115+00_BUKR_GMW_PB1_F-208.csv (origin=83034-1)
  [OK] Metrics calculated\n
[5/207] Processing: 83034-1 HB007PB01 BE115+00_INST_B_GMW_PB1_F-495.csv (origin=83034-1)
  [OK] Metrics calculated\n
[6/207] Processing: 83034-1 HB008PB01 BE115+00_BIT_GMW_PB1_F-549.csv (origin=83034-1)
  [OK] Metrics calculated\n
[7/207] Processing: 83034-1 HB009PB01 BE0280+00_BUKR_GMW_PB1_F-350.csv (origin=83034-1)
  [OK] Metrics calculated\n
[8/207] Processing: 83034-1 HB010PB01 BE0280+00_BIKR_GMW_PB1_F-346.csv (origin=83034-1)
  [OK] Metrics calculated\n
[9/207] Processing: 83034-1 H

  [OK] Metrics calculated\n
[21/207] Processing: 86349-1 HB009PB01 HB_BE0098+80_BIKR_GMW_PB1_F-248.csv (origin=86349-1)
  [OK] Metrics calculated\n
[22/207] Processing: 86349-1 HB010PB01 HB_BE0098+81_BIB_GMW_PB1_F-188.csv (origin=86349-1)
  [ERR] No valid head data\n
[23/207] Processing: 86349-1 HB011PB01 HB_BE0147+68_BUKR_GMW_PB1_F-247.csv (origin=86349-1)
  [OK] Metrics calculated\n
[24/207] Processing: 86349-1 HB013PB01 HB_BE0147+68_BIB_GMW_PB1_F-654.csv (origin=86349-1)
  [OK] Metrics calculated\n
[25/207] Processing: 86349-1 HB014PB01 HB_BE0147+67_BIT_GMW_PB1_F-650.csv (origin=86349-1)
  [OK] Metrics calculated\n
[26/207] Processing: 86349-1 HB015PB01 HB_BE0166+94_BIKR_GMW_PB1_F-238.csv (origin=86349-1)
  [OK] Metrics calculated\n
[27/207] Processing: 86349-1 HB017-APB01 HB_BE0166+95_BIB_GMW_PB1_F-602.csv (origin=86349-1)
  [OK] Metrics calculated\n
[28/207] Processing: 86349-1 HB018PB01 HB_BE0166+95_BIT_GMW_PB1_F-599.csv (origin=86349-1)


  [OK] Metrics calculated\n
[29/207] Processing: 86349-1 HB019PB01 HB_BE0188+24_BUKR_GMW_PB1_F-352.csv (origin=86349-1)
  [OK] Metrics calculated\n
[30/207] Processing: 86349-1 HB021PB01 HB_BE0188+23_BIB_GMW_PB1_F-689.csv (origin=86349-1)
  [OK] Metrics calculated\n
[31/207] Processing: 86349-1 HB022PB01 HB_BE0188+23_BIT_GMW_PB1_F-702.csv (origin=86349-1)
  [OK] Metrics calculated\n
[32/207] Processing: 86349-1 HB023PB01 HB_BE0195+80_BUKR_GMW_PB1_F-207.csv (origin=86349-1)
  [OK] Metrics calculated\n
[33/207] Processing: 86349-1 HB025PB01 HB_BE0195+81_BIB_GMW_PB1_F-566.csv (origin=86349-1)
  [OK] Metrics calculated\n
[34/207] Processing: 86349-1 HB026PB01 HB_BE0195+81_BIT_GMW_PB1_F-601.csv (origin=86349-1)
  [OK] Metrics calculated\n
[35/207] Processing: 86349-1 HB027PB01 HB_BE0213+10_BIKR_GMW_PB1_F-253.csv (origin=86349-1)
  [ERR] No valid head data\n
[36/207] Processing: 86349-1 HB029PB01 HB_BE0213+10_BIB_GMW_PB1_F-599.csv (origin=86349-1)


  [OK] Metrics calculated\n
[37/207] Processing: 86349-1 HB030PB01 HB_BE0213+10_BIT_GMW_PB1_F-681.csv (origin=86349-1)
  [OK] Metrics calculated\n
[38/207] Processing: 86349-1 HB031PB01 HB_BE0235+10_BIKR_GMW_PB1_F-235.csv (origin=86349-1)
  [OK] Metrics calculated\n
[39/207] Processing: 86349-1 HB033PB01 HB_BE0235+10_BIT_GMW_PB1_F-648.csv (origin=86349-1)
  [OK] Metrics calculated\n
[40/207] Processing: 86349-1 HB034PB01 HB_BE0242+8_BUKR_GMW_PB1_F-250.csv (origin=86349-1)
  [OK] Metrics calculated\n
[41/207] Processing: 86349-1 HB036PB01 HB_BE0242+7_BIT_GMW_PB1_F-654.csv (origin=86349-1)
  [OK] Metrics calculated\n
[42/207] Processing: 86349-1 HB037PB01 HB_BE0254+96_BUKR_GMW_PB1_F-198.csv (origin=86349-1)
  [OK] Metrics calculated\n
[43/207] Processing: 86349-1 HB039PB01 HB_BE0254+96_BIB_GMW_PB1_F-443.csv (origin=86349-1)
  [OK] Metrics calculated\n
[44/207] Processing: 86349-1 HB040PB01 HB_BE0254+96_BIT_GMW_PB1_F-454.csv (origin=86349-1)


  [OK] Metrics calculated\n
[45/207] Processing: 86349-1 HB041PB01 HB_BE0263+75_BUKR_GMW_PB1_F-248.csv (origin=86349-1)
  [OK] Metrics calculated\n
[46/207] Processing: 86349-1 HB046PB01 HB_BE0328+3_BIB_GMW_PB1_F-654.csv (origin=86349-1)
  [OK] Metrics calculated\n
[47/207] Processing: 86349-1 HB048PB01 HB_BE0377+32_BUKR_GMW_PB1_F-197.csv (origin=86349-1)
  [OK] Metrics calculated\n
[48/207] Processing: 86349-1 HB050PB01 HB_BE0377+32_BIB_GMW_PB1_F-553.csv (origin=86349-1)
  [OK] Metrics calculated\n
[49/207] Processing: 86349-1 HB051PB01 HB_BE0377+32_BIT_GMW_PB1_F-447.csv (origin=86349-1)
  [OK] Metrics calculated\n
[50/207] Processing: 86349-1 MB001PB01 B_BE0072+3_BUKR_GMW_PB1_F-385.csv (origin=86349-1)
  [OK] Metrics calculated\n
[51/207] Processing: 86349-1 MB001PB02 B_BE0072+3_BUKR_GMW_PB2_F-750.csv (origin=86349-1)
  [OK] Metrics calculated\n
[52/207] Processing: 86349-1 MB003PB01 B_BE0072+3_BIT_GMW_PB1_F-519.csv (origin=86349-1)


...

  [OK] Metrics calculated\n
[158/207] Processing: 88111-1 HB015PB01 HB_SC0054+90_INSD_GMW_PB1_F-773.csv (origin=88111-1)
  [OK] Metrics calculated\n
[159/207] Processing: 88111-1 HB016PB01 HB_SC0081+0_BUKR_GMW_PB1_F-249.csv (origin=88111-1)
  [OK] Metrics calculated\n
[160/207] Processing: 88111-1 HB018PB01 HB_SC0081+0_BIT_GMW_PB1_F-406.csv (origin=88111-1)
  [OK] Metrics calculated\n
[161/207] Processing: 88111-1 HB020PB01 HB_SC0096+0_BUKR_GMW_PB1_F-273.csv (origin=88111-1)
  [OK] Metrics calculated\n
[162/207] Processing: 88111-1 HB022PB01 HB_SC0096+0_BIT_GMW_PB1_F-523.csv (origin=88111-1)
  [OK] Metrics calculated\n
[163/207] Processing: 88111-1 HB024PB01 HB_SC0121+0_BUKR_GMW_PB1_F-247.csv (origin=88111-1)
  [OK] Metrics calculated\n
[164/207] Processing: 88111-1 HB026PB01 HB_SC0121+0_INBIB_GMW_PB1_F-418.csv (origin=88111-1)
  [OK] Metrics calculated\n
[165/207] Processing: 88111-1 HB027PB01 HB_SC0121+0_KRBIB_GMW_PB1_F-410.csv (origin=88111-1)
  [OK] Metrics calculated\n
[166/207] P

  [OK] Metrics calculated\n
[172/207] Processing: 88111-1 HB037PB01 HB_SC0179+50_BIT_GMW_PB1_F-483.csv (origin=88111-1)
  [OK] Metrics calculated\n
[173/207] Processing: 88111-1 HB038PB01 HB_SC0179+50_BIT_GMW_PB1_F-244.csv (origin=88111-1)
  [OK] Metrics calculated\n
[174/207] Processing: 88111-1 HB040PB01 HB_SC0200+50_INBIB_GMW_PB1_F-489.csv (origin=88111-1)
  [OK] Metrics calculated\n
[175/207] Processing: 88111-1 HB042PB01 HB_SC0219+50_BIKR_GMW_PB1_F-325.csv (origin=88111-1)
  [ERR] Error processing 88111-1 HB042PB01 HB_SC0219+50_BIKR_GMW_PB1_F-325.csv: unsupported operand type(s) for -: 'str' and 'str'\n
[176/207] Processing: 88111-1 HB044PB01 HB_SC0219+50_BIT_GMW_PB1_F-520.csv (origin=88111-1)
  [OK] Metrics calculated\n
[177/207] Processing: 88111-1 HB046PB01 HB_SC0234+0_BIKR_GMW_PB1_F-246.csv (origin=88111-1)
  [OK] Metrics calculated\n
[178/207] Processing: 88111-1 HB047PB01 HB_SC0234+0_BIT_GMW_PB1_F-418.csv (origin=88111-1)
  [OK] Metrics calculated\n
[179/207] Processing: 881

  [OK] Metrics calculated\n
[186/207] Processing: 88111-1 HB057PB01 HB_SC0341+0_BIKR_GMW_PB1_F-285.csv (origin=88111-1)
  [ERR] Error processing 88111-1 HB057PB01 HB_SC0341+0_BIKR_GMW_PB1_F-285.csv: unsupported operand type(s) for -: 'str' and 'str'\n
[187/207] Processing: 88111-1 HB059PB01 HB_SC0341+0_INSD_GMW_PB1_F-673.csv (origin=88111-1)
  [ERR] Error processing 88111-1 HB059PB01 HB_SC0341+0_INSD_GMW_PB1_F-673.csv: unsupported operand type(s) for -: 'str' and 'str'\n
[188/207] Processing: 88111-1 HB060-APB01 HB_SC0029+0_BITA_GMW_PB1_F-306.csv (origin=88111-1)
  [OK] Metrics calculated\n
[189/207] Processing: 88111-1 HB060-APB02 HB_SC0029+0_BITA_GMW_PB2_F-756.csv (origin=88111-1)
  [OK] Metrics calculated\n
[190/207] Processing: 88111-1 MB002PB01 B_SC0009+0_BIKR_GMW_PB1_F-297.csv (origin=88111-1)
  [ERR] Error processing 88111-1 MB002PB01 B_SC0009+0_BIKR_GMW_PB1_F-297.csv: unsupported operand type(s) for -: 'str' and 'str'\n
[191/207] Processing: 88111-1 MB012PB01 B_SC0054+90_BUKR_G

  [OK] Metrics calculated\n
[200/207] Processing: 88111-1 MB036PB01 B_SC0179+50_BIKR_GMW_PB1_F-335.csv (origin=88111-1)
  [OK] Metrics calculated\n
[201/207] Processing: 88111-1 MB039PB01 B_SC0200+50_BIKR_GMW_PB1_F-306.csv (origin=88111-1)
  [OK] Metrics calculated\n
[202/207] Processing: 88111-1 MB043PB01 B_SC0219+50_BUKR_GMW_PB1_F-331.csv (origin=88111-1)
  [ERR] Error processing 88111-1 MB043PB01 B_SC0219+50_BUKR_GMW_PB1_F-331.csv: unsupported operand type(s) for -: 'str' and 'str'\n
[203/207] Processing: 88111-1 MB043PB02 B_SC0219+50_BUKR_GMW_PB2_F-851.csv (origin=88111-1)
  [ERR] Error processing 88111-1 MB043PB02 B_SC0219+50_BUKR_GMW_PB2_F-851.csv: unsupported operand type(s) for -: 'str' and 'str'\n
[204/207] Processing: 88111-1 MB045PB01 B_SC0234+0_BUKR_GMW_PB1_F-317.csv (origin=88111-1)
  [OK] Metrics calculated\n
[205/207] Processing: 88111-1 MB045PB02 B_SC0234+0_BUKR_GMW_PB2_F-717.csv (origin=88111-1)
  [OK] Metrics calculated\n
[206/207] Processing: 88111-1 MB056PB01 B_SC03

[OK] Report saved to: D:\Users\jvanruitenbeek\data_validation\output_data\validation_report_wiertsema.xlsx
\nSummary Statistics:
       Source Origin                                        Filename  \
count            199                                             199   
unique             5                                             199   
top          87074-1  83034-1 HB001PB01 BE0049+00_BUKR_GMW_PB1_F-229   
freq              64                                               1   
mean             NaN                                             NaN   
min              NaN                                             NaN   
25%              NaN                                             NaN   
50%              NaN                                             NaN   
75%              NaN                                             NaN   
max              NaN                                             NaN   
std              NaN                                             NaN   

      